# CRM AI project


The AI converts that request into SQL, your Python program executes the query, and the result is returned to you.


1. User asks a question
```
"Show all customers from Kerala"
```
2. Python reads database metadata
```
Tables, columns, data types and foreign-key relationships
```

3. Gemini generates SQL
```
The question and schema are sent to the AI model
```

4. Python validates the SQL
```
The notebook blocks queries containing DROP or TRUNCATE
```

5. MySQL executes the query
```
SELECT retrieves data; other statements may change data
```

6. Python returns the result
```
A DataFrame, success message, or AI-generated error explanation
```

In [ ]:
!pip install pymysql 
!pip install plotly
!pip install nbformat

In [ ]:
!pip install google-genai
!pip install groq

In [ ]:
import os
from google import genai
import pymysql
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from groq import Groq

In [ ]:
load_dotenv()
conn = pymysql.connect(
    host=os.getenv("MYSQL_HOST", "127.0.0.1"),
    port=int(os.getenv("MYSQL_PORT", "3306")),
    user=os.getenv("MYSQL_USER", "root"),
    password=os.environ["MYSQL_PASSWORD"],
    database=os.getenv("MYSQL_DATABASE", "crm_ai"),
)

cursor = conn.cursor()
print("Connected successfully!")

## Schema Definition & Gemini Model Setup

### Initial manual text representation of the database schema

In [ ]:
SCHEMA = """
DATABASE : crm_ai

TABLE: customers

customer_id INT PRIMARY KEY AUTO_INCREMENT
name VARCHAR(100)
email VARCHAR(100)
phone VARCHAR(20)
city VARCHAR(50)
state VARCHAR(50)
created_at DATETIME

TABLE: products

product_id INT PRIMARY KEY AUTO_INCREMENT
product_name VARCHAR(100)
category VARCHAR(50)
price DECIMAL(10,2)

TABLE: orders

order_id INT PRIMARY KEY AUTO_INCREMENT
customer_id INT FOREIGN KEY REFERENCES customers(customer_id)
order_date DATE
total_amount DECIMAL(10,2)
status VARCHAR(20)

TABLE: order_items

item_id INT PRIMARY KEY AUTO_INCREMENT
order_id INT FOREIGN KEY REFERENCES orders(order_id)
product_id INT FOREIGN KEY REFERENCES products(product_id)
quantity INT
amount DECIMAL(10,2)

TABLE: leads

lead_id INT PRIMARY KEY AUTO_INCREMENT
lead_name VARCHAR(100)
source VARCHAR(50)
status VARCHAR(30)
assigned_to VARCHAR(50)

TABLE: employees

employee_id INT PRIMARY KEY AUTO_INCREMENT
employee_name VARCHAR(100)
department VARCHAR(50)
email VARCHAR(100)

RELATIONSHIPS:

customers.customer_id
       |
       |
orders.customer_id

orders.order_id
       |
       |
order_items.order_id

products.product_id
       |
       |
order_items.product_id
"""

### SDK client and lists all available Gemini models

In [ ]:
load_dotenv()
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# List available models
for model in client.models.list():
    print(f"Name: {model.name}")
    print(f"Display Name: {model.display_name}")
    print("-" * 40)

### Initialize Gemini Client

In [ ]:
# The client automatically picks up GEMINI_API_KEY from environment variables
#client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# Generate content directly using the client
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Hello, Gemini!"
)

print(response.text)

### Dynamic Database Metadata Extraction

In [ ]:
cursor.execute("""
SELECT
table_name,
column_name,
data_type
FROM information_schema.columns
WHERE table_schema='crm_ai'
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

In [ ]:
cursor.execute("""
SELECT
table_name,
column_name
FROM information_schema.key_column_usage
WHERE table_schema='crm_ai'
AND constraint_name='PRIMARY'
""")

pk_rows = cursor.fetchall()

print(pk_rows)

In [ ]:
def get_schema():

    query = """
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_schema='crm_ai'
    ORDER BY table_name, ordinal_position
    """

    cursor.execute(query)

    rows = cursor.fetchall()

    schema = ""

    current_table = None

    for table, column, dtype, nullable in rows:

        if current_table != table:

            schema += f"\n\nTABLE: {table}\n"

            current_table = table

        schema += (
            f"   {column} "
            f"({dtype}) "
            f"NULLABLE={nullable}\n"
        )

    return schema

In [ ]:
def get_relationships():

    query = """
    SELECT
        table_name,
        column_name,
        referenced_table_name,
        referenced_column_name
    FROM information_schema.key_column_usage
    WHERE table_schema='crm_ai'
    AND referenced_table_name IS NOT NULL
    """

    cursor.execute(query)

    rows = cursor.fetchall()

    relationships = "\n\nRELATIONSHIPS:\n"

    for row in rows:

        relationships += (
            f"{row[0]}.{row[1]}"
            f" --> "
            f"{row[2]}.{row[3]}\n"
        )

    return relationships

In [ ]:
def build_context():

    schema = get_schema()

    relationships = get_relationships()

    context = schema + relationships

    return context

In [ ]:
build_context()

In [ ]:
def generate_sql(user_question):
    schema_context = build_context()
    prompt = f"""
    You are an expert MySQL developer.
    DATABASE SCHEMA:
    {schema_context}
    RULES:
    1. Return ONLY executable MySQL.
    2. No markdown, no explanations, no comments.
    3. Use joins when required.
    4. Never generate DROP or TRUNCATE.
    USER QUESTION:
    {user_question}
    """
    response = client.models.generate_content(model="gemini-3.5-flash", contents=prompt)
    sql = response.text.replace("```sql", "").replace("```", "").strip()
    return sql

In [ ]:
def validate_sql(sql):
    blocked_keywords = ["drop", "truncate"]
    sql_lower = sql.lower()
    for keyword in blocked_keywords:
        if keyword in sql_lower:
            raise Exception(f"Blocked Query Detected: {keyword}")
    return True

In [ ]:
import pandas as pd

In [ ]:
def explain_error(user_query, sql, error):
    prompt = f"""
    You are a CRM assistant.
    User Request: {user_query}
    Generated SQL: {sql}
    Database Error: {error}
    Explain the issue in simple business language and suggest what to do next.
    Do not show technical database jargon.
    """

    # Correct SDK call
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )
    return response.text

In [ ]:
def execute_sql_with_retry(sql, user_query, max_retries=2):
    """Executes SQL with an agentic self-correction loop if errors occur."""
    current_sql = sql
    
    for attempt in range(max_retries + 1):
        try:
            validate_sql(current_sql)
            cursor.execute(current_sql)
            
            if current_sql.lower().startswith("select"):
                rows = cursor.fetchall()
                columns = [d[0] for d in cursor.description]
                return pd.DataFrame(rows, columns=columns)
            
            conn.commit()
            return "Operation Successful"
            
        except Exception as e:
            error_msg = str(e)
            
            # If we've exhausted retries, fallback to friendly explanation
            if attempt == max_retries:
                return explain_error(user_query, current_sql, error_msg)
            
            # Self-Correction prompt to Gemini
            correction_prompt = f"""
            You are an expert MySQL developer fixing a broken query.
            
            Original User Request: {user_query}
            Failed SQL Query: {current_sql}
            MySQL Error: {error_msg}
            
            Please analyze the error, fix the SQL query, and return ONLY executable MySQL code without markdown or explanations.
            """
            
            response = client.models.generate_content(
                model="gemini-3.5-flash",
                contents=correction_prompt
            )
            
            current_sql = (
                response.text
                .replace("```sql", "")
                .replace("```", "")
                .strip()
            )
            print(f"Retrying with corrected SQL (Attempt {attempt + 1}):\n{current_sql}")

In [ ]:
def classify_intent(user_query):
    """Classifies user input to ensure it falls within CRM boundaries."""
    prompt = f"""
    You are a security and intent router for a CRM AI assistant. 
    The database contains customers, products, orders, order items, leads, and employees.
    
    Analyze the user query: "{user_query}"
    
    Is this query related to querying, updating, or managing CRM data, business metrics, or entities?
    Answer ONLY with 'YES' or 'NO'.
    """
    
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )
    
    return response.text.strip().upper()

In [ ]:
def auto_visualize(df, user_query):
    """Automatically detects data patterns and renders a Plotly chart if appropriate."""
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return
    
    # Check if DataFrame has at least one categorical/date column and one numeric column
    cat_cols = df.select_dtypes(include=['object', 'category', 'datetime']).columns
    num_cols = df.select_dtypes(include=['number']).columns
    
    if len(cat_cols) >= 1 and len(num_cols) >= 1:
        x_col = cat_cols[0]
        y_col = num_cols[0]
        
        # Avoid plotting IDs as metrics
        if 'id' not in x_col.lower() and len(df) > 1:
            print(f"\nRendering interactive chart for: {user_query}")
            
            # Use bar chart for categories, line chart if it looks like time-series
            if 'date' in x_col.lower() or 'created' in x_col.lower():
                fig = px.line(df, x=x_col, y=y_col, markers=True, title=f"Trend: {user_query}")
            else:
                fig = px.bar(df, x=x_col, y=y_col, text_auto=True, title=f"Insights: {user_query}")
                
            fig.update_layout(template="plotly_dark", xaxis_tickangle=-45)
            fig.show()

In [ ]:
def ask_crm_secure(question):
    """Enhanced wrapper with intent checking and self-healing execution."""
    intent = classify_intent(question)
    
    if "NO" in intent:
        return "I'm your CRM assistant. I can only help you query, manage, or analyze customer, product, order, and lead data."
    
    sql = generate_sql(question)
    print("Generated SQL:\n", sql)
    
    result = execute_sql_with_retry(sql, question)
    return result

### Test Execution

In [ ]:
query = "Show all customers from Kerala"
result = ask_crm_secure(query)

display(result)


In [ ]:
auto_visualize(result, query)

In [ ]:
ask_crm_secure(
    "Show top 5 customers by revenue"
)

In [ ]:
ask_crm_secure(
    "List customers who purchased Laptop Pro"
)

In [ ]:
ask_crm_secure(
    "Show pending orders"
)

In [ ]:
ask_crm_secure(
    "Delete all lost leads"
)

In [ ]:
ask_crm_secure(
    "add one row"
)